<a href="https://colab.research.google.com/github/1Poras1/INDIA-SPACE-LAB-ISL-2026/blob/Figure-Eight-Drone-Trajectory-Tracking/Task3_Figure8_Drone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creative Bonus Task — Figure-Eight Drone Trajectory Tracking

In [11]:
#Libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon, Circle
from IPython.display import HTML, display
from ipywidgets import interact, FloatSlider

### Step 1 — Draw a drone shape
A tiny quadcopter icon: an X-shaped body with 4 rotor circles

In [4]:
def create_drone(x, y, theta, scale=0.6):
    # Body: a simple X/cross frame
    arm = np.array([
        [ 1,  1],
        [-1, -1],
        [ 0,  0],
        [ 1, -1],
        [-1,  1],
    ]) * scale * 0.5

    R = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    rotated_arm = arm @ R.T
    rotated_arm[:, 0] += x
    rotated_arm[:, 1] += y

    # Rotor centers at the 4 arm tips
    tips = np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]]) * scale * 0.5
    rotated_tips = tips @ R.T
    rotated_tips[:, 0] += x
    rotated_tips[:, 1] += y

    return rotated_arm, rotated_tips

### Step 2 — Define the figure-eight path
This uses a **Lissajous curve** (`x = A sin(ωt)`, `y = (A/2) sin(2ωt)`), which naturally traces a figure-eight. Because we have the exact equations, we can also compute the desired *velocity* and *acceleration* analytically

In [5]:
def figure8_path(t, A=5.0, w=0.5):
    # Position
    xd = A * np.sin(w * t)
    yd = (A / 2) * np.sin(2 * w * t)
    # Velocity (1st derivative) -- feedforward
    vxd = A * w * np.cos(w * t)
    vyd = A * w * np.cos(2 * w * t)
    # Acceleration (2nd derivative) -- feedforward
    axd = -A * w**2 * np.sin(w * t)
    ayd = -2 * A * w**2 * np.sin(2 * w * t)
    return xd, yd, vxd, vyd, axd, ayd

### Step 3 — The controller + drone physics
The drone is modeled as a point mass with air drag

In [6]:
def simulate_drone(kp=10.0, kd=6.0, ki=0.1, drag=0.5,
                    A=5.0, w=0.5,
                    wind_amp=0.3, wind_freq=0.7,
                    dt=0.02, cycles=2):

    T = cycles * 2 * np.pi / w
    t = np.arange(0, T, dt)

    x, y = 0.0, 0.0
    vx, vy = 0.0, 0.0
    ix, iy = 0.0, 0.0  # integral error accumulators

    traj_x, traj_y = [], []
    des_x, des_y = [], []
    wind_x_hist, wind_y_hist = [], []
    headings = []
    speeds = []

    for ti in t:
        xd, yd, vxd, vyd, axd, ayd = figure8_path(ti, A, w)

        ex, ey = xd - x, yd - y
        evx, evy = vxd - vx, vyd - vy
        ix += ex * dt
        iy += ey * dt

        # Time-varying wind disturbance
        wind_x = wind_amp * np.sin(wind_freq * ti)
        wind_y = wind_amp * np.cos(0.8 * wind_freq * ti)

        # Controller: feedforward + PD + I
        ux = axd + kp * ex + kd * evx + ki * ix
        uy = ayd + kp * ey + kd * evy + ki * iy

        # Physics: drag opposes motion, wind pushes the drone around
        ax = ux - drag * vx + wind_x
        ay = uy - drag * vy + wind_y

        vx += ax * dt
        vy += ay * dt
        x += vx * dt
        y += vy * dt

        traj_x.append(x); traj_y.append(y)
        des_x.append(xd); des_y.append(yd)
        wind_x_hist.append(wind_x); wind_y_hist.append(wind_y)
        headings.append(np.arctan2(vy, vx))
        speeds.append(np.hypot(vx, vy))

    return dict(
        t=t, x=np.array(traj_x), y=np.array(traj_y),
        xd=np.array(des_x), yd=np.array(des_y),
        wind_x=np.array(wind_x_hist), wind_y=np.array(wind_y_hist),
        headings=np.array(headings), speeds=np.array(speeds)
    )

### Step 4 — Static plot with tracking-error readout
The actual trajectory is colour-coded by the drone's speed

In [7]:
def run_and_plot(kp=10.0, kd=6.0, ki=0.1, drag=0.5,
                  wind_amp=0.3, wind_freq=0.7, speed=0.5):

    sim = simulate_drone(kp=kp, kd=kd, ki=ki, drag=drag,
                          w=speed, wind_amp=wind_amp, wind_freq=wind_freq)

    rmse = np.sqrt(np.mean((sim['x']-sim['xd'])**2 + (sim['y']-sim['yd'])**2))

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.plot(sim['xd'], sim['yd'], '--', color='gray', linewidth=2, label='Desired Figure-8')

    sc = ax.scatter(sim['x'], sim['y'], c=sim['speeds'], cmap='plasma',
                     s=8, label='Drone Trajectory (colour = speed)')
    plt.colorbar(sc, ax=ax, label='Speed (m/s)', shrink=0.8)

    # Drone icon at the final position
    arm, tips = create_drone(sim['x'][-1], sim['y'][-1], sim['headings'][-1], scale=1.2)
    ax.add_patch(Polygon(arm, closed=False, fill=False, edgecolor='black', linewidth=2))
    for tx, ty in tips:
        ax.add_patch(Circle((tx, ty), 0.25, color='black'))

    # Wind field
    ax.quiver(sim['x'][-1], sim['y'][-1], sim['wind_x'][-1], sim['wind_y'][-1],
               color='dodgerblue', scale=10, width=0.005, label='Wind (current)')

    ax.set_title(f'Figure-8 Tracking | Kp={kp} Kd={kd} Ki={ki} | RMSE={rmse:.3f} m')
    ax.set_xlabel('X Position (m)')
    ax.set_ylabel('Y Position (m)')
    ax.grid(True, alpha=0.4)
    ax.legend(loc='lower right')
    ax.set_aspect('equal')
    plt.show()

### Step 5 — Tune it live
Drag the sliders and watch the RMSE and trajectory shape change in real time.

In [8]:
interact(
    run_and_plot,
    kp=FloatSlider(value=10.0, min=0.5, max=15.0, step=0.5, description='Kp (Gain)'),
    kd=FloatSlider(value=6.0, min=0.0, max=10.0, step=0.5, description='Kd (Damping)'),
    ki=FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, description='Ki (Integral)'),
    drag=FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description='Drag'),
    wind_amp=FloatSlider(value=0.3, min=0.0, max=2.0, step=0.1, description='Wind Amp'),
    wind_freq=FloatSlider(value=0.7, min=0.1, max=3.0, step=0.1, description='Wind Freq'),
    speed=FloatSlider(value=0.5, min=0.2, max=1.0, step=0.05, description='Path Speed (ω)'),
)

interactive(children=(FloatSlider(value=10.0, description='Kp (Gain)', max=15.0, min=0.5, step=0.5), FloatSlid…

<function __main__.run_and_plot(kp=10.0, kd=6.0, ki=0.1, drag=0.5, wind_amp=0.3, wind_freq=0.7, speed=0.5)>

### Step 6 — Animate it
This renders a smooth animation: the drone icon banks with its heading, a fading trail shows where it's been and a live readout in the corner shows the instantaneous tracking error.

In [12]:
def animate_drone_figure8(kp=10.0, kd=6.0, ki=0.1, drag=0.5,
                           wind_amp=0.3, wind_freq=0.7, speed=0.5,
                           trail_len=60):

    sim = simulate_drone(kp=kp, kd=kd, ki=ki, drag=drag,
                          w=speed, wind_amp=wind_amp, wind_freq=wind_freq)
    n = len(sim['t'])

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.set_xlim(sim['xd'].min() - 2, sim['xd'].max() + 2)
    ax.set_ylim(sim['yd'].min() - 2, sim['yd'].max() + 2)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.4)
    ax.set_title('Animated Figure-8 Drone Tracking')
    ax.set_xlabel('X Position (m)')
    ax.set_ylabel('Y Position (m)')

    ax.plot(sim['xd'], sim['yd'], '--', color='gray', linewidth=1.5, label='Desired Path')
    trail_line, = ax.plot([], [], color='crimson', linewidth=2, alpha=0.8, label='Recent Trail')

    drone_outline, = ax.plot([], [], color='black', linewidth=2)
    rotor_dots = ax.scatter([], [], color='black', s=30, zorder=5)

    error_text = ax.text(0.02, 0.96, '', transform=ax.transAxes,
                          fontsize=10, va='top',
                          bbox=dict(boxstyle='round', fc='white', alpha=0.8))

    ax.legend(loc='lower right')

    def update(frame):
        i = frame
        start = max(0, i - trail_len)
        trail_line.set_data(sim['x'][start:i+1], sim['y'][start:i+1])

        arm, tips = create_drone(sim['x'][i], sim['y'][i], sim['headings'][i], scale=1.0)
        drone_outline.set_data(arm[:, 0], arm[:, 1])
        rotor_dots.set_offsets(tips)

        err = np.hypot(sim['x'][i]-sim['xd'][i], sim['y'][i]-sim['yd'][i])
        error_text.set_text(f'error: {err:.2f} m\nspeed: {sim["speeds"][i]:.2f} m/s')

        return trail_line, drone_outline, rotor_dots, error_text

    anim = FuncAnimation(fig, update, frames=n, interval=25, blit=False)
    plt.close(fig)
    return anim

In [13]:
# Changing the gain to whatever liked best from the sliders, then run this cell
anim = animate_drone_figure8(
    kp=10.0,
    kd=6.0,
    ki=0.1,
    drag=0.5,
    wind_amp=0.3,
    wind_freq=0.7,
    speed=0.5
)
HTML(anim.to_html5_video())